[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-4/research-assistant.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239974-lesson-4-research-assistant)

# 研究助手（Research Assistant）

## 回顾

我们已经介绍了几个重要的 LangGraph 主题：

* Memory（记忆）
* Human-in-the-loop（人机协同）
* Controllability（可控性）

现在，我们将这些概念整合起来，应对 AI 最热门的应用之一：研究自动化。

研究工作往往是交由分析师完成的繁重工作。AI 在协助这方面具有相当大的潜力。

然而，研究需要定制化：原始 LLM 输出通常不适合现实世界的决策工作流。

定制化的、基于 AI 的[研究和报告生成](https://jxnl.co/writing/2024/06/05/predictions-for-the-future-of-rag/#reports-over-rag)工作流是解决这个问题的一种有前景的方法。

## 目标

我们的目标是围绕聊天模型构建一个轻量级的多 Agent 系统，用于定制化研究流程。

`Source Selection`（来源选择）
* 用户可以为他们的研究选择任意一组输入来源。

`Planning`（规划）
* 用户提供一个主题，系统会生成一个 AI 分析师团队，每个分析师专注于一个子主题。
* 在开始研究之前，将使用 `Human-in-the-loop`（人机协同）来完善这些子主题。

`LLM Utilization`（LLM 利用）
* 每个分析师将使用选定的来源与专家 AI 进行深入访谈。
* 访谈将是一个多轮对话，以提取详细的见解，如 [STORM](https://arxiv.org/abs/2402.14207) 论文中所示。
* 这些访谈将使用带有内部状态的 `sub-graphs`（子图）来记录。

`Research Process`（研究过程）
* 专家将`并行`收集信息以回答分析师的问题。
* 所有访谈将通过 `map-reduce` 同时进行。

`Output Format`（输出格式）
* 每次访谈收集到的见解将被合成为最终报告。
* 我们将为报告使用可定制的提示，允许灵活的输出格式。

![Screenshot 2024-08-26 at 7.26.33 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb164d61c93d48e604091_research-assistant1.png)

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_deepseek langchain_community langchain_core langchain-tavily wikipedia

## Setup

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

In [ ]:
from langchain_deepseek import ChatDeepSeek
llm = ChatDeepSeek(model="deepseek-chat", temperature=0) 

我们将使用 [LangSmith](https://docs.langchain.com/langsmith/home) 进行[追踪](https://docs.langchain.com/langsmith/observability-concepts)。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 生成分析师：人机协同

创建分析师并使用人机协同对其进行审查。

In [ ]:
from typing import List
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

class Analyst(BaseModel):
    affiliation: str = Field(
        description="分析师的主要隶属机构。",
    )
    name: str = Field(
        description="分析师的姓名。"
    )
    role: str = Field(
        description="分析师在当前主题背景下的角色。",
    )
    description: str = Field(
        description="分析师关注点、关切和动机的描述。",
    )
    @property
    def persona(self) -> str:
        return f"Name: {self.name}\nRole: {self.role}\nAffiliation: {self.affiliation}\nDescription: {self.description}\n"

class Perspectives(BaseModel):
    analysts: List[Analyst] = Field(
        description="包含分析师及其角色和隶属机构的完整列表。",
    )

class GenerateAnalystsState(TypedDict):
    topic: str # 研究主题
    max_analysts: int # 分析师数量
    human_analyst_feedback: str # 人类反馈
    analysts: List[Analyst] # 正在提问的分析师

In [ ]:
from IPython.display import Image, display
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

analyst_instructions="""You are tasked with creating a set of AI analyst personas. Follow these instructions carefully:

1. First, review the research topic:
{topic}
        
2. Examine any editorial feedback that has been optionally provided to guide creation of the analysts: 
        
{human_analyst_feedback}
    
3. Determine the most interesting themes based upon documents and / or feedback above.
                    
4. Pick the top {max_analysts} themes.

5. Assign one analyst to each theme."""

def create_analysts(state: GenerateAnalystsState):
    
    """ 创建分析师 """
    
    topic=state['topic']
    max_analysts=state['max_analysts']
    human_analyst_feedback=state.get('human_analyst_feedback', '')
        
    # 强制结构化输出
    structured_llm = llm.with_structured_output(Perspectives)

    # 系统消息
    system_message = analyst_instructions.format(topic=topic,
                                                            human_analyst_feedback=human_analyst_feedback, 
                                                            max_analysts=max_analysts)

    # 生成问题
    analysts = structured_llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content="Generate the set of analysts.")])
    
    # 将分析师列表写入状态
    return {"analysts": analysts.analysts}

def human_feedback(state: GenerateAnalystsState):
    """ 空操作节点，在此处中断 """
    pass

def should_continue(state: GenerateAnalystsState):
    """ 返回下一个要执行的节点 """

    # 检查是否有用户反馈
    human_analyst_feedback=state.get('human_analyst_feedback', None)
    if human_analyst_feedback:
        return "create_analysts"
    
    # 否则终止
    return END

# 添加节点和边
builder = StateGraph(GenerateAnalystsState)
builder.add_node("create_analysts", create_analysts)
builder.add_node("human_feedback", human_feedback)
builder.add_edge(START, "create_analysts")
builder.add_edge("create_analysts", "human_feedback")
builder.add_conditional_edges("human_feedback", should_continue, ["create_analysts", END])

# 编译
memory = MemorySaver()
graph = builder.compile(interrupt_before=['human_feedback'], checkpointer=memory)

# 查看
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
# 输入
max_analysts = 3 
topic = "The benefits of adopting LangGraph as an agent framework"
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream({"topic":topic,"max_analysts":max_analysts,}, thread, stream_mode="values"):
    # 审查
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50)

In [ ]:
# Get state and look at next node
state = graph.get_state(thread)
state.next

In [ ]:
# 我们现在以 human_feedback 节点的身份更新状态
graph.update_state(thread, {"human_analyst_feedback": 
                            "Add in someone from a startup to add an entrepreneur perspective"}, as_node="human_feedback")

In [ ]:
# 继续图的执行
for event in graph.stream(None, thread, stream_mode="values"):
    # 审查
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50)

In [ ]:
# 如果我们满意了，就不提供反馈
further_feedack = None
graph.update_state(thread, {"human_analyst_feedback": 
                            further_feedack}, as_node="human_feedback")

In [ ]:
# 继续图的执行直到结束
for event in graph.stream(None, thread, stream_mode="updates"):
    print("--Node--")
    node_name = next(iter(event.keys()))
    print(node_name)

In [ ]:
final_state = graph.get_state(thread)
analysts = final_state.values.get('analysts')

In [ ]:
final_state.next

In [ ]:
for analyst in analysts:
    print(f"Name: {analyst.name}")
    print(f"Affiliation: {analyst.affiliation}")
    print(f"Role: {analyst.role}")
    print(f"Description: {analyst.description}")
    print("-" * 50) 

## 进行访谈

### 生成问题

分析师将向专家提问。

In [ ]:
import operator
from typing import  Annotated
from langgraph.graph import MessagesState

class InterviewState(MessagesState):
    max_num_turns: int # 对话轮数
    context: Annotated[list, operator.add] # 来源文档
    analyst: Analyst # 正在提问的分析师
    interview: str # 访谈文本记录
    sections: list # 最终键，我们在外层状态中为 Send() API 复制该键

class SearchQuery(BaseModel):
    search_query: str = Field(None, description="用于检索的搜索查询。")

In [ ]:
question_instructions = """You are an analyst tasked with interviewing an expert to learn about a specific topic. 

Your goal is boil down to interesting and specific insights related to your topic.

1. Interesting: Insights that people will find surprising or non-obvious.
        
2. Specific: Insights that avoid generalities and include specific examples from the expert.

Here is your topic of focus and set of goals: {goals}
        
Begin by introducing yourself using a name that fits your persona, and then ask your question.

Continue to ask questions to drill down and refine your understanding of the topic.
        
When you are satisfied with your understanding, complete the interview with: "Thank you so much for your help!"

Remember to stay in character throughout your response, reflecting the persona and goals provided to you."""

def generate_question(state: InterviewState):
    """ 用于生成问题的节点 """

    # 获取状态
    analyst = state["analyst"]
    messages = state["messages"]

    # 生成问题
    system_message = question_instructions.format(goals=analyst.persona)
    question = llm.invoke([SystemMessage(content=system_message)]+messages)
        
    # 将消息写入状态
    return {"messages": [question]}

### 生成答案：并行化

专家将从多个来源并行收集信息以回答问题。

例如，我们可以使用：

* 特定网站，例如通过 [`WebBaseLoader`](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base)
* 索引文档，例如通过 [RAG](https://docs.langchain.com/oss/python/langchain/retrieval)
* 网络搜索
* Wikipedia 搜索

你可以尝试不同的网络搜索工具，例如 [Tavily](https://tavily.com/)。

In [ ]:
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("TAVILY_API_KEY")

In [ ]:
# Web search tool
from langchain_tavily import TavilySearch  # 在 1.0 中更新

tavily_search = TavilySearch(max_results=3)

In [ ]:
# Wikipedia search tool
from langchain_community.document_loaders import WikipediaLoader

现在，我们创建节点来搜索网络和 Wikipedia。

我们还将创建一个节点来回答分析师的问题。

最后，我们将创建节点来保存完整的访谈记录并撰写访谈摘要（"section"）。

In [ ]:
from langchain_core.messages import get_buffer_string

# 搜索查询编写
search_instructions = SystemMessage(content=f"""You will be given a conversation between an analyst and an expert. 

Your goal is to generate a well-structured query for use in retrieval and / or web-search related to the conversation.
        
First, analyze the full conversation.

Pay particular attention to the final question posed by the analyst.

Convert this final question into a well-structured web search query""")

def search_web(state: InterviewState):
    
    """ 从网络搜索中检索文档 """

    # 搜索查询
    structured_llm = llm.with_structured_output(SearchQuery)
    search_query = structured_llm.invoke([search_instructions]+state['messages'])
    
    # 搜索
    data = tavily_search.invoke({"query": search_query.search_query})
    search_docs = data.get("results", data)

     # 格式化
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
            for doc in search_docs
        ]
    )

    return {"context": [formatted_search_docs]} 

def search_wikipedia(state: InterviewState):
    
    """ 从 Wikipedia 检索文档 """

    # 搜索查询
    structured_llm = llm.with_structured_output(SearchQuery)
    search_query = structured_llm.invoke([search_instructions]+state['messages'])
    
    # 搜索
    search_docs = WikipediaLoader(query=search_query.search_query, 
                                  load_max_docs=2).load()

     # 格式化
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}"/>\n{doc.page_content}\n</Document>'
            for doc in search_docs
        ]
    )

    return {"context": [formatted_search_docs]} 

answer_instructions = """You are an expert being interviewed by an analyst.

Here is analyst area of focus: {goals}. 
        
You goal is to answer a question posed by the interviewer.

To answer question, use this context:
        
{context}

When answering questions, follow these guidelines:
        
1. Use only the information provided in the context. 
        
2. Do not introduce external information or make assumptions beyond what is explicitly stated in the context.

3. The context contain sources at the topic of each individual document.

4. Include these sources your answer next to any relevant statements. For example, for source # 1 use [1]. 

5. List your sources in order at the bottom of your answer. [1] Source 1, [2] Source 2, etc
        
6. If the source is: <Document source="assistant/docs/llama3_1.pdf" page="7"/>' then just list: 
        
[1] assistant/docs/llama3_1.pdf, page 7 
        
And skip the addition of the brackets as well as the Document source preamble in your citation."""

def generate_answer(state: InterviewState):
    
    """ 用于回答问题的节点 """

    # 获取状态
    analyst = state["analyst"]
    messages = state["messages"]
    context = state["context"]

    # 回答问题
    system_message = answer_instructions.format(goals=analyst.persona, context=context)
    answer = llm.invoke([SystemMessage(content=system_message)]+messages)
            
    # 将消息命名为来自专家
    answer.name = "expert"
    
    # 追加到状态
    return {"messages": [answer]}

def save_interview(state: InterviewState):
    
    """ 保存访谈记录 """

    # 获取消息
    messages = state["messages"]
    
    # 将访谈转换为字符串
    interview = get_buffer_string(messages)
    
    # 保存到 interviews 键
    return {"interview": interview}

def route_messages(state: InterviewState, 
                   name: str = "expert"):

    """ 在问题和答案之间进行路由 """
    
    # 获取消息
    messages = state["messages"]
    max_num_turns = state.get('max_num_turns',2)

    # 检查专家回答的数量
    num_responses = len(
        [m for m in messages if isinstance(m, AIMessage) and m.name == name]
    )

    # 如果专家回答的轮数已超过最大限制，则结束
    if num_responses >= max_num_turns:
        return 'save_interview'

    # 此路由器在每对问答后运行
    # 获取最后提出的问题，检查是否标志对话结束
    last_question = messages[-2]
    
    if "Thank you so much for your help" in last_question.content:
        return 'save_interview'
    return "ask_question"

section_writer_instructions = """You are an expert technical writer. 
            
Your task is to create a short, easily digestible section of a report based on a set of source documents.

1. Analyze the content of the source documents: 
- The name of each source document is at the start of the document, with the <Document tag.
        
2. Create a report structure using markdown formatting:
- Use ## for the section title
- Use ### for sub-section headers
        
3. Write the report following this structure:
a. Title (## header)
b. Summary (### header)
c. Sources (### header)

4. Make your title engaging based upon the focus area of the analyst: 
{focus}

5. For the summary section:
- Set up summary with general background / context related to the focus area of the analyst
- Emphasize what is novel, interesting, or surprising about insights gathered from the interview
- Create a numbered list of source documents, as you use them
- Do not mention the names of interviewers or experts
- Aim for approximately 400 words maximum
- Use numbered sources in your report (e.g., [1], [2]) based on information from source documents
        
6. In the Sources section:
- Include all sources used in your report
- Provide full links to relevant websites or specific document paths
- Separate each source by a newline. Use two spaces at the end of each line to create a newline in Markdown.
- It will look like:

### Sources
[1] Link or Document name
[2] Link or Document name

7. Be sure to combine sources. For example this is not correct:

[3] https://ai.meta.com/blog/meta-llama-3-1/
[4] https://ai.meta.com/blog/meta-llama-3-1/

There should be no redundant sources. It should simply be:

[3] https://ai.meta.com/blog/meta-llama-3-1/
        
8. Final review:
- Ensure the report follows the required structure
- Include no preamble before the title of the report
- Check that all guidelines have been followed"""

def write_section(state: InterviewState):

    """ 用于撰写 section 的节点 """

    # 获取状态
    interview = state["interview"]
    context = state["context"]
    analyst = state["analyst"]
   
    # 使用从访谈中收集的来源文档（context）或访谈本身（interview）来撰写 section
    system_message = section_writer_instructions.format(focus=analyst.description)
    section = llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content=f"Use this source to write your section: {context}")]) 
                
    # 追加到状态
    return {"sections": [section.content]}

# 添加节点和边
interview_builder = StateGraph(InterviewState)
interview_builder.add_node("ask_question", generate_question)
interview_builder.add_node("search_web", search_web)
interview_builder.add_node("search_wikipedia", search_wikipedia)
interview_builder.add_node("answer_question", generate_answer)
interview_builder.add_node("save_interview", save_interview)
interview_builder.add_node("write_section", write_section)

# 流程
interview_builder.add_edge(START, "ask_question")
interview_builder.add_edge("ask_question", "search_web")
interview_builder.add_edge("ask_question", "search_wikipedia")
interview_builder.add_edge("search_web", "answer_question")
interview_builder.add_edge("search_wikipedia", "answer_question")
interview_builder.add_conditional_edges("answer_question", route_messages,['ask_question','save_interview'])
interview_builder.add_edge("save_interview", "write_section")
interview_builder.add_edge("write_section", END)

# 访谈
memory = MemorySaver()
interview_graph = interview_builder.compile(checkpointer=memory).with_config(run_name="Conduct Interviews")

# 查看
display(Image(interview_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Pick one analyst
analysts[0]

这里，我们运行访谈，传入与我们的主题相关的 llama3.1 论文索引。

In [ ]:
from IPython.display import Markdown
messages = [HumanMessage(f"So you said you were writing an article on {topic}?")]
thread = {"configurable": {"thread_id": "1"}}
interview = interview_graph.invoke({"analyst": analysts[0], "messages": messages, "max_num_turns": 2}, thread)
Markdown(interview['sections'][0])

### 并行化访谈：Map-Reduce

我们通过 `Send()` API（map 步骤）并行化访谈。

我们将它们组合成 reduce 步骤中的报告正文。

### 最终化

我们添加一个最终步骤来撰写最终报告的引言和结论。

In [ ]:
import operator
from typing import List, Annotated
from typing_extensions import TypedDict

class ResearchGraphState(TypedDict):
    topic: str # 研究主题
    max_analysts: int # 分析师数量
    human_analyst_feedback: str # 人类反馈
    analysts: List[Analyst] # 正在提问的分析师
    sections: Annotated[list, operator.add] # Send() API 键
    introduction: str # 最终报告的引言
    content: str # 最终报告的内容
    conclusion: str # 最终报告的结论
    final_report: str # 最终报告

In [ ]:
from langgraph.types import Send # 在 1.0 中更新
def initiate_all_interviews(state: ResearchGraphState):
    """ 这是 "map" 步骤，我们使用 Send API 运行每个访谈子图 """    

    # 检查是否有用户反馈
    human_analyst_feedback=state.get('human_analyst_feedback')
    if human_analyst_feedback:
        # 返回 create_analysts
        return "create_analysts"

    # 否则通过 Send() API 并行启动访谈
    else:
        topic = state["topic"]
        return [Send("conduct_interview", {"analyst": analyst,
                                           "messages": [HumanMessage(
                                               content=f"So you said you were writing an article on {topic}?"
                                           )
                                                       ]}) for analyst in state["analysts"]]

report_writer_instructions = """You are a technical writer creating a report on this overall topic: 

{topic}
    
You have a team of analysts. Each analyst has done two things: 

1. They conducted an interview with an expert on a specific sub-topic.
2. They write up their finding into a memo.

Your task: 

1. You will be given a collection of memos from your analysts.
2. Think carefully about the insights from each memo.
3. Consolidate these into a crisp overall summary that ties together the central ideas from all of the memos. 
4. Summarize the central points in each memo into a cohesive single narrative.

To format your report:
 
1. Use markdown formatting. 
2. Include no pre-amble for the report.
3. Use no sub-heading. 
4. Start your report with a single title header: ## Insights
5. Do not mention any analyst names in your report.
6. Preserve any citations in the memos, which will be annotated in brackets, for example [1] or [2].
7. Create a final, consolidated list of sources and add to a Sources section with the `## Sources` header.
8. List your sources in order and do not repeat.

[1] Source 1
[2] Source 2

Here are the memos from your analysts to build your report from: 

{context}"""

def write_report(state: ResearchGraphState):
    # 完整的 sections 集合
    sections = state["sections"]
    topic = state["topic"]

    # 将所有 sections 拼接在一起
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])
    
    # 将 sections 总结为最终报告
    system_message = report_writer_instructions.format(topic=topic, context=formatted_str_sections)    
    report = llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content=f"Write a report based upon these memos.")]) 
    return {"content": report.content}

intro_conclusion_instructions = """You are a technical writer finishing a report on {topic}

You will be given all of the sections of the report.

You job is to write a crisp and compelling introduction or conclusion section.

The user will instruct you whether to write the introduction or conclusion.

Include no pre-amble for either section.

Target around 100 words, crisply previewing (for introduction) or recapping (for conclusion) all of the sections of the report.

Use markdown formatting. 

For your introduction, create a compelling title and use the # header for the title.

For your introduction, use ## Introduction as the section header. 

For your conclusion, use ## Conclusion as the section header.

Here are the sections to reflect on for writing: {formatted_str_sections}"""

def write_introduction(state: ResearchGraphState):
    # 完整的 sections 集合
    sections = state["sections"]
    topic = state["topic"]

    # 将所有 sections 拼接在一起
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])
    
    # 将 sections 总结为最终报告
    
    instructions = intro_conclusion_instructions.format(topic=topic, formatted_str_sections=formatted_str_sections)    
    intro = llm.invoke([instructions]+[HumanMessage(content=f"Write the report introduction")]) 
    return {"introduction": intro.content}

def write_conclusion(state: ResearchGraphState):
    # 完整的 sections 集合
    sections = state["sections"]
    topic = state["topic"]

    # 将所有 sections 拼接在一起
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])
    
    # 将 sections 总结为最终报告
    
    instructions = intro_conclusion_instructions.format(topic=topic, formatted_str_sections=formatted_str_sections)    
    conclusion = llm.invoke([instructions]+[HumanMessage(content=f"Write the report conclusion")]) 
    return {"conclusion": conclusion.content}

def finalize_report(state: ResearchGraphState):
    """ 这是 "reduce" 步骤，我们收集所有 sections，组合它们，并基于它们撰写引言/结论 """
    # 保存完整的最终报告
    content = state["content"]
    if content.startswith("## Insights"):
        content = content.strip("## Insights")
    if "## Sources" in content:
        try:
            content, sources = content.split("\n## Sources\n")
        except:
            sources = None
    else:
        sources = None

    final_report = state["introduction"] + "\n\n---\n\n" + content + "\n\n---\n\n" + state["conclusion"]
    if sources is not None:
        final_report += "\n\n## Sources\n" + sources
    return {"final_report": final_report}

# 添加节点和边
builder = StateGraph(ResearchGraphState)
builder.add_node("create_analysts", create_analysts)
builder.add_node("human_feedback", human_feedback)
builder.add_node("conduct_interview", interview_builder.compile())
builder.add_node("write_report",write_report)
builder.add_node("write_introduction",write_introduction)
builder.add_node("write_conclusion",write_conclusion)
builder.add_node("finalize_report",finalize_report)

# 逻辑
builder.add_edge(START, "create_analysts")
builder.add_edge("create_analysts", "human_feedback")
builder.add_conditional_edges("human_feedback", initiate_all_interviews, ["create_analysts", "conduct_interview"])
builder.add_edge("conduct_interview", "write_report")
builder.add_edge("conduct_interview", "write_introduction")
builder.add_edge("conduct_interview", "write_conclusion")
builder.add_edge(["write_conclusion", "write_report", "write_introduction"], "finalize_report")
builder.add_edge("finalize_report", END)

# 编译
memory = MemorySaver()
graph = builder.compile(interrupt_before=['human_feedback'], checkpointer=memory)
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

让我们问一个关于 LangGraph 的开放式问题。

In [ ]:
# 输入
max_analysts = 3 
topic = "The benefits of adopting LangGraph as an agent framework"
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream({"topic":topic,
                           "max_analysts":max_analysts}, 
                          thread, 
                          stream_mode="values"):
    
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50)

In [ ]:
# 我们现在以 human_feedback 节点的身份更新状态
graph.update_state(thread, {"human_analyst_feedback": 
                                "Add in the CEO of gen ai native startup"}, as_node="human_feedback")

In [ ]:
# 检查
for event in graph.stream(None, thread, stream_mode="values"):
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50)

In [ ]:
# 确认我们满意
graph.update_state(thread, {"human_analyst_feedback": 
                            None}, as_node="human_feedback")

In [ ]:
# 继续
for event in graph.stream(None, thread, stream_mode="updates"):
    print("--Node--")
    node_name = next(iter(event.keys()))
    print(node_name)

In [ ]:
from IPython.display import Markdown
final_state = graph.get_state(thread)
report = final_state.values.get('final_report')
Markdown(report)

我们可以查看追踪记录：

https://smith.langchain.com/public/2933a7bb-bcef-4d2d-9b85-cc735b22ca0c/r